# Dataset Preparation

Downloads the multilingual code comments dataset from HuggingFace and prepares it for the evaluation pipeline.

**Outputs:**
- `prepared_data_{Language}/final_batch_pipeline.json` - Ground truth with expert annotations
- `prepared_data_{Language}/final_batch_judge.json` - Input for judge LLM (no annotations)

## 1. Configuration

In [1]:
import json
import re
from pathlib import Path
from typing import Dict, List, Any
from datasets import load_dataset

# Dataset configuration
DATASET_NAME = "AISE-TUDelft/multilingual-code-comments-fixed-2"
LANGUAGES = ['English', 'Dutch', 'Chinese', 'Greek', 'Polish']
BASE_DIR = Path.cwd()

print(f"Dataset: {DATASET_NAME}")
print(f"Languages: {', '.join(LANGUAGES)}")

c:\projects\EMSE-Babel\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: AISE-TUDelft/multilingual-code-comments-fixed-2
Languages: English, Dutch, Chinese, Greek, Polish


## 2. Transformation Functions

Convert HuggingFace dataset format to pipeline JSON structure.

In [2]:
def clean_eos_tokens(text: str) -> str:
    """Remove end-of-sequence tokens and everything after them."""
    if not text:
        return text
    
    eos_patterns = [
        r'<\|file_separator\|>', r'<\|endoftext\|>', r'<file_sep>',
        r'<fim_suffix>', r'<fim_middle>', r'<fim_pad>',
        r'<\|fim_suffix\|>', r'<\|fim_middle\|>',
        r'</s>', r'<eos>', r'<\|end\|>',
    ]
    
    earliest_pos = len(text)
    for pattern in eos_patterns:
        match = re.search(pattern, text)
        if match and match.start() < earliest_pos:
            earliest_pos = match.start()
    
    return text[:earliest_pos].rstrip()


def extract_model_names(sample: Dict[str, Any]) -> List[str]:
    """Extract sorted list of model names from sample keys."""
    return sorted([
        key.replace('predicted_comment_', '')
        for key in sample.keys()
        if key.startswith('predicted_comment_')
    ])


def to_pipeline_format(sample: Dict[str, Any], language: str) -> Dict[str, Any]:
    """Convert sample to pipeline format (includes expert annotations)."""
    model_names = extract_model_names(sample)
    
    model_predictions = []
    for model in model_names:
        error_codes_raw = sample.get(f'error_codes_{model}')
        error_codes = []
        if error_codes_raw and error_codes_raw != 'None':
            if isinstance(error_codes_raw, str):
                error_codes = [c.strip() for c in error_codes_raw.split(',')]
            elif isinstance(error_codes_raw, list):
                error_codes = error_codes_raw
        
        predicted = sample.get(f'predicted_comment_{model}', '')
        
        model_predictions.append({
            "model_name": model,
            "predicted_comment": clean_eos_tokens(predicted),
            "masked_code": sample.get(f'masked_data_{model}', ''),
            "expert_accuracy": sample.get(f'expert_accuracy_{model}', ''),
            "error_codes": error_codes
        })
    
    return {
        "metadata": {
            "file_id": sample['file_id'],
            "repository": sample['repo'],
            "file_path": sample['path'],
            "language": language
        },
        "code_context": {
            "source_code": sample['content'],
            "original_comment": sample['original_comment']
        },
        "model_predictions": model_predictions
    }


def to_judge_format(sample: Dict[str, Any], language: str) -> Dict[str, Any]:
    """Convert sample to judge format (no expert annotations)."""
    model_names = extract_model_names(sample)
    
    model_predictions = []
    for model in model_names:
        predicted = sample.get(f'predicted_comment_{model}', '')
        model_predictions.append({
            "model_name": model,
            "predicted_comment": clean_eos_tokens(predicted),
            "masked_code": sample.get(f'masked_data_{model}', '')
        })
    
    return {
        "metadata": {
            "file_id": sample['file_id'],
            "language": language
        },
        "code_context": {
            "source_code": sample['content'],
            "original_comment": sample['original_comment']
        },
        "model_predictions": model_predictions
    }


print("Functions defined.")

Functions defined.


## 3. Download and Process All Languages

For each language: download from HuggingFace, transform, save to `prepared_data_{Language}/`.

In [3]:
for language in LANGUAGES:
    print(f"\n{'='*50}")
    print(f"{language}")
    print(f"{'='*50}")
    
    # Download
    dataset = load_dataset(DATASET_NAME, language)
    samples = dataset['train']
    print(f"Downloaded {len(samples)} samples")
    
    # Transform
    pipeline_data = [to_pipeline_format(s, language) for s in samples]
    judge_data = [to_judge_format(s, language) for s in samples]
    
    # Save
    output_dir = BASE_DIR / f"prepared_data_{language}"
    output_dir.mkdir(exist_ok=True)
    
    with open(output_dir / "final_batch_pipeline.json", 'w', encoding='utf-8') as f:
        json.dump(pipeline_data, f, indent=2, ensure_ascii=False)
    
    with open(output_dir / "final_batch_judge.json", 'w', encoding='utf-8') as f:
        json.dump(judge_data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved to {output_dir}")

print(f"\n{'='*50}")
print("Done!")
print(f"{'='*50}")


English


c:\projects\EMSE-Babel\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ziemm\.cache\huggingface\hub\datasets--AISE-TUDelft--multilingual-code-comments-fixed-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 500/500 [00:00<00:00, 8184.71 examples/s]


Downloaded 500 samples
Saved to c:\projects\EMSE-Babel\prepared_data_English

Dutch


Generating train split: 100%|██████████| 500/500 [00:00<00:00, 6582.44 examples/s]


Downloaded 500 samples
Saved to c:\projects\EMSE-Babel\prepared_data_Dutch

Chinese


...

Downloaded 500 samples
Saved to c:\projects\EMSE-Babel\prepared_data_Chinese

Greek


Generating train split: 100%|██████████| 500/500 [00:00<00:00, 7977.88 examples/s]


Downloaded 500 samples
Saved to c:\projects\EMSE-Babel\prepared_data_Greek

Polish


Generating train split: 100%|██████████| 500/500 [00:00<00:00, 11651.04 examples/s]


Downloaded 500 samples
Saved to c:\projects\EMSE-Babel\prepared_data_Polish

Done!


## 4. Verify Output

Quick check that files were created correctly.

In [4]:
print("Output files:")
for language in LANGUAGES:
    output_dir = BASE_DIR / f"prepared_data_{language}"
    for f in sorted(output_dir.glob("final_batch_*.json")):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.relative_to(BASE_DIR)} ({size_mb:.2f} MB)")

# Sample verification
test_file = BASE_DIR / "prepared_data_English" / "final_batch_judge.json"
with open(test_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"\nEnglish sample count: {len(data)}")
print(f"Models per sample: {len(data[0]['model_predictions'])}")
print(f"Sample file_id: {data[0]['metadata']['file_id']}")

Output files:
  prepared_data_English\final_batch_judge.json (11.66 MB)
  prepared_data_English\final_batch_judge_fixed.json (11.66 MB)
  prepared_data_English\final_batch_pipeline.json (11.90 MB)
  prepared_data_English\final_batch_pipeline_fixed.json (11.90 MB)
  prepared_data_Dutch\final_batch_judge.json (13.38 MB)
  prepared_data_Dutch\final_batch_judge_fixed.json (13.39 MB)
  prepared_data_Dutch\final_batch_pipeline.json (13.67 MB)
  prepared_data_Dutch\final_batch_pipeline_fixed.json (13.67 MB)
  prepared_data_Chinese\final_batch_judge.json (12.04 MB)
  prepared_data_Chinese\final_batch_judge_fixed.json (12.03 MB)
  prepared_data_Chinese\final_batch_pipeline.json (12.36 MB)
  prepared_data_Chinese\final_batch_pipeline_fixed.json (12.35 MB)
  prepared_data_Greek\final_batch_judge.json (14.08 MB)
  prepared_data_Greek\final_batch_judge_fixed.json (14.08 MB)
  prepared_data_Greek\final_batch_pipeline.json (14.39 MB)
  prepared_data_Greek\final_batch_pipeline_fixed.json (14.39 MB)
  